# Baseline Radar – Learning Notebook

Tento notebook vysvětluje krok za krokem, co dělá `baseline_radar.py`, a vizualizuje celé řešení jako studijní materiál pro soutěžící.

## Cíle notebooku
- pochopit datový formát radar tasku (`.mat.pt`, tvar `7 x 50 x 181`),
- vizualizovat vstupní kanály i cílové masky,
- ukázat průchod modelem `BaselineCNN` a predikci,
- ukázat očekávaný formát submission (`submission_val.csv`, `submission_test.csv`, `submission.zip`).

## 0) Runtime requirements

Pokud ti notebook spadne na `ModuleNotFoundError`, doinstaluj závislosti:

```bash
pip install -r requirements.txt matplotlib
```


## 1) Přehled pipeline z `baseline_radar.py`

1. (Volitelně) stáhne data z IOAI repo (`--download`).
2. Použije složky:
   - `training_set`: `data/ioai-2025/Individual-Contest/Radar/training_set`
   - `val_set`: `data/ioai-2025/Individual-Contest/Radar/Solution/validation_set`
   - `test_set`: `data/ioai-2025/Individual-Contest/Radar/Solution/test_set`
3. Načte tréninkové tensory `7 x 50 x 181`:
   - prvních 6 kanálů = vstup `x`,
   - poslední kanál = labely `y` v rozsahu `[-1, 3]`.
4. Posune labely na `[0, 4]` kvůli `CrossEntropyLoss`.
5. Natrénuje jednoduché CNN (`6 -> 16 -> 32 -> 5`).
6. Vygeneruje predikce pro val/test a uloží je zpět jako `[-1, 3]`.
7. Vytvoří odevzdání do ZIP.

In [ ]:
from pathlib import Path
import sys
import importlib.util

# Volitelné auto-doinstalování matplotlib (užitečné v čistém prostředí)
if importlib.util.find_spec("matplotlib") is None:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib"])

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

# Najdi root repozitáře tak, aby šel importovat baseline_radar.py
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
repo_root = None
for cand in candidates:
    if (cand / "baseline_radar.py").exists():
        repo_root = cand
        break
if repo_root is None:
    raise FileNotFoundError("Nenašel jsem baseline_radar.py. Spusť notebook z rootu repa nebo z ai-perception/.")
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from baseline_radar import (
    RadarTrainDataset,
    BaselineCNN,
    run_epoch,
)

plt.rcParams["figure.figsize"] = (14, 8)


In [ ]:
# Pevně nastavené cesty podle zadání
TRAIN_DIR = Path("data/ioai-2025/Individual-Contest/Radar/training_set")
VAL_DIR = Path("data/ioai-2025/Individual-Contest/Radar/Solution/validation_set")
TEST_DIR = Path("data/ioai-2025/Individual-Contest/Radar/Solution/test_set")

if TRAIN_DIR.exists() and VAL_DIR.exists() and TEST_DIR.exists():
    train_dir, val_dir, test_dir = TRAIN_DIR, VAL_DIR, TEST_DIR
    print("Train:", train_dir)
    print("Val  :", val_dir)
    print("Test :", test_dir)
    use_real_data = True
else:
    print("Některá z požadovaných složek neexistuje. Notebook dál poběží na syntetickém příkladu.")
    print("Expected TRAIN:", TRAIN_DIR)
    print("Expected VAL  :", VAL_DIR)
    print("Expected TEST :", TEST_DIR)
    use_real_data = False


## 2) Vizualizace jednoho vzorku z tréninku

Každý tréninkový soubor má tvar `7 x 50 x 181`:
- `x = tensor[:6]` (6 radarových kanálů),
- `y = tensor[6]` (segmentační maska).

In [ ]:
if use_real_data:
    train_ds = RadarTrainDataset(train_dir)
    x, y_shifted = train_ds[0]   # y už je po posunu: [0,4]
    y = y_shifted - 1            # zpět na [-1,3] kvůli interpretaci
    source = "real sample"
else:
    # Syntetický fallback, aby notebook šel spustit i bez datasetu
    x = torch.randn(6, 50, 181)
    y = torch.randint(low=-1, high=4, size=(50, 181))
    source = "synthetic sample"

print(f"Source: {source}")
print("x shape:", tuple(x.shape))
print("y shape:", tuple(y.shape))
print("y unique classes:", sorted(torch.unique(y).tolist()))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    im = ax.imshow(x[i].numpy(), aspect="auto", cmap="viridis")
    ax.set_title(f"Input channel {i}")
    ax.set_xlabel("Angle bins (181)")
    ax.set_ylabel("Range bins (50)")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle("Radar input channels (6)")
plt.tight_layout()
plt.show()

In [ ]:
# Maska tříd: -1..3
plt.figure(figsize=(12,4))
plt.imshow(y.numpy(), aspect="auto", cmap="tab10", vmin=-1, vmax=3)
plt.colorbar(label="Class ID")
plt.title("Ground-truth segmentation mask (classes -1..3)")
plt.xlabel("Angle bins (181)")
plt.ylabel("Range bins (50)")
plt.show()

## 3) Mini trénink přímo v notebooku

Tato část ukazuje, že **trénink i vizualizace běží přímo v ipynb**.

- Pokud existují reálná data, trénuje se na `training_set`.
- Pokud data chybí, použije se syntetický dataset (aby šel notebook spustit vždy).

Pro rychlý learning demo běží jen pár epoch.


In [ ]:
class SyntheticRadarTrainDataset(Dataset):
    def __init__(self, n_samples: int = 64):
        self.n_samples = n_samples

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        x = torch.randn(6, 50, 181)
        y = torch.randint(low=0, high=5, size=(50, 181), dtype=torch.long)  # CE labels in [0,4]
        return x, y


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BaselineCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

if use_real_data:
    train_ds = RadarTrainDataset(train_dir)
    print(f"Training on real dataset: {len(train_ds)} samples")
else:
    train_ds = SyntheticRadarTrainDataset(n_samples=64)
    print("Training on synthetic dataset fallback: 64 samples")

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)

epochs = 2 if use_real_data else 1
history_loss, history_score = [], []

for epoch in range(1, epochs + 1):
    train_loss, train_score = run_epoch(model, train_loader, loss_fn, device, optimizer)
    history_loss.append(train_loss)
    history_score.append(train_score)
    print(f"Epoch {epoch:02d}/{epochs} | train_loss={train_loss:.4f} train_score={train_score:.4f}")

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history_loss, marker="o")
plt.title("Training loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.subplot(1,2,2)
plt.plot(history_score, marker="o")
plt.title("Training IOAI score")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.ylim(0, 1)

plt.tight_layout()
plt.show()


## 4) Jak funguje model `BaselineCNN`

Model vrací logity tvaru `5 x 50 x 181`:
- 5 tříd odpovídá mapování `[-1, 3] -> [0, 4]`.

In [ ]:
model.eval()

with torch.no_grad():
    logits = model(x.unsqueeze(0))      # [1, 6, 50, 181] -> [1, 5, 50, 181]
    pred_shifted = torch.argmax(logits, dim=1).squeeze(0)  # [50, 181], values 0..4
    pred = pred_shifted - 1             # zpět na -1..3

print("logits shape:", tuple(logits.shape))
print("pred shape:", tuple(pred.shape))
print("pred unique classes:", sorted(torch.unique(pred).tolist()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(y.numpy(), aspect="auto", cmap="tab10", vmin=-1, vmax=3)
axes[0].set_title("Ground truth")
axes[0].set_xlabel("Angle bins")
axes[0].set_ylabel("Range bins")

axes[1].imshow(pred.numpy(), aspect="auto", cmap="tab10", vmin=-1, vmax=3)
axes[1].set_title("BaselineCNN prediction (untrained/random if synthetic)")
axes[1].set_xlabel("Angle bins")
axes[1].set_ylabel("Range bins")

plt.tight_layout()
plt.show()

## 5) Očekávaný output z úlohy (submission formát)

`baseline_radar.py` po doběhnutí uloží:
- `outputs/baseline_cnn.pt`
- `outputs/submission_val.csv`
- `outputs/submission_test.csv`
- `outputs/submission.zip`

CSV má řádky ve formátu:
- první sloupec: `filename`
- dalších `50*181 = 9050` sloupců: `pixel_0 ... pixel_9049`
- hodnoty pixelů: třídy v rozsahu `[-1, 3]`

In [ ]:
# Mini ukázka "jak vypadá" 1 řádek submission CSV
example_filename = "000001.mat.pt"
example_pred = pred.flatten().tolist()   # délka 9050

header = ["filename"] + [f"pixel_{i}" for i in range(50*181)]
row = [example_filename] + example_pred

print("Header length:", len(header))
print("Row length   :", len(row))
print("First 12 header columns:", header[:12])
print("First 12 row values:", row[:12])

## 6) Referenční průběh tréninku (co čekat v terminálu)

Při spuštění baseline skriptu typicky uvidíš něco jako:

```text
Train: data/ioai-2025/Individual-Contest/Radar/training_set
Val:   data/ioai-2025/Individual-Contest/Radar/Solution/validation_set
Test:  data/ioai-2025/Individual-Contest/Radar/Solution/test_set
Epoch 01/8 | train_loss=... train_score=...
...
Epoch 08/8 | train_loss=... train_score=...
Saved model: outputs/baseline_cnn.pt
Saved submissions: outputs/submission_val.csv, outputs/submission_test.csv
Saved archive: outputs/submission.zip
```

> Přesná čísla loss/score se liší podle seedu, HW a verze knihoven.

## 7) Doporučení pro soutěžící

- baseline je hlavně **kontrolní minimum** (fungující end-to-end pipeline),
- pro lepší výsledek zkus:
  - hlubší architekturu (např. U-Net-like decoder),
  - class weighting / focal loss (kvůli silné převaze backgroundu),
  - augmentace a lepší validaci,
  - scheduler + delší trénink.

Hodně štěstí! 🚀